<a href="https://colab.research.google.com/github/pedroManuelP/C-digos-em-Python/blob/main/Lista3_3_SistControle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [149]:
import numpy as np
from numpy.polynomial import Polynomial
import scipy as sp
import re

def polinomioCaracteristico(A):
  A = -A; s = Polynomial([0, 1])
  if(A.shape[0] == 3):
    plus = (s+A[0][0])*(s+A[1][1])*(s+A[2][2]) + (A[0][1]*A[1][2]*A[2][0]) + (A[0][2]*A[1][0]*A[2][1])
    minus = (A[2][0]*(s+A[1][1])*A[0][2]) + (A[2][1]*A[1][2]*(s+A[0][0])) + ((s+A[2][2])*A[1][0]*A[0][1])
    poly = plus-minus; polos = poly.roots()
  elif(A.shape[0] == 2):
    plus = (s+A[0][0])*(s+A[1][1])
    minus = (A[1][0]*A[0][1])
    poly = plus-minus; polos = poly.roots()
  elif(A.shape[0] == 1):
    poly = s+A[0][0]; polos = poly.roots()
  else:
    print("Não é possível calcular o polinômio característico.")
  polos = np.round(polos, 3)
  #print("Polinômio característico: " + str(poly))
  print("Polos: " + str(polos))
  for i in range(len(polos)):
    if(np.abs(polos[i].real) > 0):
      print("O sistema é instável.")
      return
  print("O sistema é estável.")

def matrizControlabilidade(A, B):
  n = A.shape[0]
  U = np.zeros((n,n))
  for i in range(n):
    U[:,i:i+1] = (np.linalg.matrix_power(A,i))@B
  return U

def matrizObservabilidade(A, C):
  n = A.shape[0]
  V = np.zeros((n,n))
  for i in range(n):
    V[i:i+1, :] = C@(np.linalg.matrix_power(A,i))
  return V

def realimentaçãoDeEstado(A, U, s):
  theta = Polynomial.fromroots(s)
  #print('\n'+str(theta))
  coeficientes = (theta.coef).real

  n = A.shape[0]; qc = np.zeros((n,n))
  for i in range(n+1):
    qc += coeficientes[i]*(np.linalg.matrix_power(A,i))
  invU = np.linalg.inv(U)
  linha = np.zeros((1,n)); linha[0][-1] = 1
  return -linha@(invU@qc)

def observadorDeEstado(A, V, s):
  theta = Polynomial.fromroots(s)
  #print('\n'+str(theta))
  coeficientes = (theta.coef).real

  n = A.shape[0]; ql = np.zeros((n,n))
  for i in range(n+1):
    ql += coeficientes[i]*(np.linalg.matrix_power(A,i))
  invV = np.linalg.inv(V)
  coluna = np.zeros((n,1)); coluna[-1][0] = 1
  return ql@invV@coluna

def matrizExpandida(G, H):
  # Matriz expandida para um seguidor de referência do tipo degrau unitário
  n = np.shape(G)[0]; Ge = np.zeros((n+1, n+1));
  Ge[0:n, 0:n] = G; Ge[0:n, n:n+1] = H;
  He = np.zeros((n+1, 1)); He[-1, 0] = 1;
  print(); print("Ge = "+str(Ge)); print("He = "+str(He));
  return Ge, He

def realimentaçãoDeEstado(A, U, s):
  theta = Polynomial.fromroots(s)
  #print('\n'+str(theta))
  coeficientes = (theta.coef).real

  n = A.shape[0]; qc = np.zeros((n,n))
  for i in range(n+1):
    qc += coeficientes[i]*(np.linalg.matrix_power(A,i))
  invU = np.linalg.inv(U)
  linha = np.zeros((1,n)); linha[0][-1] = 1
  return np.round(linha@(invU@qc), 3)

def matrizReferencia(G, H, C, Ke):
  n = np.shape(G)[0];

  a = np.zeros([1,n+1]); a[0,-1] = 1;

  b = np.zeros((n+1, n+1));
  b[0:n, 0:n] = G-np.eye(n); b[0:n, n:n+1] = H;
  b[n, 0:n] = np.matmul(C,G); b[n, n:n+1] = np.matmul(C,H);

  return np.round(np.matmul((Ke+a),np.linalg.inv(b)), 3)


In [152]:
A = np.array([[0, 1],
              [0, -1]])
B = np.array([[0],
              [1]])
C = np.array([[3, -2]])
print("Matrizes do sistema contínuo:");
print("A = " + str(A)); print("B = " + str(C)); print("C = " + str(C))
sys_c = sp.signal.StateSpace(A, B, C)


T = 1; sys_d = sys_c.to_discrete(T)
G = np.round(sys_d.A, 3); H = np.round(sys_d.B, 3);
print("\nDiscretizando o sistema para um período de amostragem de "+str(T)+" segundo(s), obtemos as seguintes matrizes");
print("G = " + str(G)); print("H = " + str(H)); print("C = " + str(C))
print(); polinomioCaracteristico(G);

Ge, He = matrizExpandida(G, H);
Wce = matrizControlabilidade(Ge, He); print("\nWce = " + str(Wce));
Ke = realimentaçãoDeEstado(Ge, Wce, [complex(0.5,0.2), complex(0.5,-0.2), -0.1]); print("Ke = " + str(Ke))

Kr = matrizReferencia(G, H, C, Ke); print("\nKr = " + str(Kr));

Matrizes do sistema contínuo:
A = [[ 0  1]
 [ 0 -1]]
B = [[ 3 -2]]
C = [[ 3 -2]]

Discretizando o sistema para um período de amostragem de 1 segundo(s), obtemos as seguintes matrizes
G = [[1.    0.632]
 [0.    0.368]]
H = [[0.368]
 [0.632]]
C = [[ 3 -2]]

Polos: [0.368 1.   ]
O sistema é instável.

Ge = [[1.    0.632 0.368]
 [0.    0.368 0.632]
 [0.    0.    0.   ]]
He = [[0.]
 [0.]
 [1.]]

Wce = [[0.       0.368    0.767424]
 [0.       0.632    0.232576]
 [1.       0.       0.      ]]
Ke = [[0.505 0.437 0.468]]

Kr = [[1.737 1.354 0.168]]


In [42]:
A = np.array([[0, 1],
              [0, -1]])
B = np.array([[0],
              [1]])
C = np.array([[3, -2]])
print("Matrizes do sistema contínuo:");
print("A = " + str(A)); print("B = " + str(C)); print("C = " + str(C))
sys_c = sp.signal.StateSpace(A, B, C)


T = 1; sys_d = sys_c.to_discrete(T)
G = np.round(sys_d.A, 3); H = np.round(sys_d.B, 3);
print("\nDiscretizando o sistema para um período de amostragem de "+str(T)+" segundo(s), obtemos as seguintes matrizes");
print("G = " + str(G)); print("H = " + str(H)); print("C = " + str(C))
print(); polinomioCaracteristico(G);

print(); Wc = matrizControlabilidade(G, H); print("Wc = " + str(Wc));
if(np.linalg.matrix_rank(Wc) == np.shape(G)[1]):
  print("O sistema é controlável.");
else:
  print("O sistema não é controlável.");

print(); Wo = matrizObservabilidade(G, C); print("Wo = " + str(Wo));
if(np.linalg.matrix_rank(Wo) == np.shape(G)[1]):
  print("O sistema é observável.");
else:
  print("O sistema não é observável.");

z = np.array([complex(0.25,0.5), complex(0.25,-0.5)]); print("\nPolos do realimentador de estados: "+str(z))
K = realimentaçãoDeEstado(G, Wc, z); print("K = " + str(K))
z = np.array([-0.5, 0.5]); print("\nPolos do observador de estados: "+str(z))
L = observadorDeEstado(G, Wo, z); print("L = " + str(L))

Matrizes do sistema contínuo:
A = [[ 0  1]
 [ 0 -1]]
B = [[ 3 -2]]
C = [[ 3 -2]]

Discretizando o sistema para um período de amostragem de 1 segundo(s), obtemos as seguintes matrizes
G = [[1.    0.632]
 [0.    0.368]]
H = [[0.368]
 [0.632]]
C = [[ 3 -2]]

Polos: [0.368 1.   ]
O sistema é instável.

Wc = [[0.368    0.767424]
 [0.632    0.232576]]
O sistema é controlável.

Wo = [[ 3.   -2.  ]
 [ 3.    1.16]]
O sistema é observável.

Gr = [[ 0.     3.    -2.   ]
 [ 0.     1.     0.632]
 [ 0.     0.     0.368]]
Hr = [[0.   ]
 [0.368]
 [0.632]]
Ur = [[ 0.         -0.16        1.83712   ]
 [ 0.368       0.767424    0.91441203]
 [ 0.632       0.232576    0.08558797]]

Polos do realimentador de estados: [0.25+0.5j 0.25-0.5j]
K = [[-1.28560127 -0.62483977]]

Polos do observador de estados: [-0.5  0.5]
L = [[ 0.43182785]
 [-0.03625823]]


In [31]:
G = np.array([[1, 0.1],
              [0.5, 0.1]]);
H = np.array([[1],
              [0.1]]);
C = np.array([[1, 1]]);
print("G = " + str(G)); print("H = " + str(H)); print("C = " + str(C))
print(); polinomioCaracteristico(G);

print(); Wc = matrizControlabilidade(G, H); print("Wc = " + str(Wc));
if(np.linalg.matrix_rank(Wc) == np.shape(G)[1]):
  print("O sistema é controlável.");
else:
  print("O sistema não é controlável.");

print(); Wo = matrizObservabilidade(G, C); print("Wo = " + str(Wo));
if(np.linalg.matrix_rank(Wo) == np.shape(G)[1]):
  print("O sistema é observável.");
else:
  print("O sistema não é observável.");

G = [[1.  0.1]
 [0.5 0.1]]
H = [[1. ]
 [0.1]]
C = [[1 1]]

Polos: [0.048 1.052]
O sistema é instável.

Wc = [[1.   1.01]
 [0.1  0.51]]
O sistema é controlável.

Wo = [[1.  1. ]
 [1.5 0.2]]
O sistema é observável.


In [96]:
n = np.shape(G)[0]; Ge = np.zeros((n+1, n+1));
Ge[0:n, 0:n] = G; Ge[0:n, n:n+1] = H;
He = np.zeros((n+1, 1)); He[-1, 0] = 1;
print(); print("Ge = "+str(Ge)); print("He = "+str(He));


Ge = [[1.    0.632 0.368]
 [0.    0.368 0.632]
 [0.    0.    0.   ]]
He = [[0.]
 [0.]
 [1.]]


In [81]:
np.shape(Ge);Ge[1,2]

np.float64(0.0)